# doorplate: a tour

Parse US addresses into components, USPS Publication 28 and RESO output, a match key and a slug. All addresses here are invented. This library formats and parses; it **does not validate** that an address exists or can receive mail.

Run it with `pip install doorplate pandas` (or build from source: `maturin develop` in `crates/doorplate-py`). The notebook runs from the `examples/` directory or the repository root.

In [ ]:
from pathlib import Path
import doorplate as uap

DATA = Path("examples/data") if Path("examples/data").exists() else Path("data")
print("library", uap.__version__)

## One address

`parse` never raises on odd input; it returns a `ParsedAddress` with `warnings` for anything it is unsure of. Only text with no address at all raises `ParseError`.

In [ ]:
a = uap.parse("123 north main street apt 4, Miami FL 33101")
print(a.street_number, a.street_dir_prefix, a.street_name, a.street_suffix)
print(a.unit_type, a.unit_number, "|", a.city, a.state, a.zip5)
print("warnings:", a.warnings)
a.to_dict()["place"]

## Output formats

The USPS form follows Publication 28; `to_reso()` uses RESO Data Dictionary 2.0 names. Both are computed when you ask.

In [ ]:
print(a.to_usps())
print(a.to_reso())

## Keys and slugs

The **key** identifies the property (`k1:` prefix; the format is a contract); `key(unit=True)` adds the unit. The **slug** is URL-safe text.

In [ ]:
print(a.key())
print(a.key(unit=True))
print(a.slug())

# Different spellings, same key:
same = [uap.parse(t).key() for t in ["123 N Main St Apt 4, Miami FL 33101",
                                     "123 north main street #4 miami, fl 33101-2200"]]
print(same, same[0] == same[1])

## Messy lines, and what the warnings say

In [ ]:
for line in (DATA / "addresses_messy.txt").read_text().splitlines():
    p = uap.parse(line)
    print(f"{line[:46]:46} -> {p.to_usps()['line1']!s:28} {p.address_type():8} {p.warnings}")

`address_type()` is one of `street`, `po_box`, `rural_route`, `military`, `general_delivery`, `dual`, `unknown`. Text it cannot place is in `unparsed`:

In [ ]:
p = uap.parse("Main St & 1st Ave, Tampa FL")
print(p.warnings, repr(p.unparsed))

## A state hint

`default_state` is a lookup hint for input with no state. It never fills `state`.

In [ ]:
print(uap.parse("281 Evelyn Dr Dothan").city)
p = uap.parse("281 Evelyn Dr Dothan", default_state="AL")
print(p.city, p.state, p.warnings)

## Fields already split

`parse_fields` takes address columns as you have them. A name-like first line goes to `attention`.

In [ ]:
p = uap.parse_fields(line1="THE GARDENS", line2="5120 PALM BEACH BLVD",
                     city="FORT MYERS", state="FL", zip="33905")
print("attention:", p.attention, "|", p.street_number, p.street_name, p.street_suffix)
print(p.slug())

## A batch

`parse_many` releases the GIL and parses in parallel. A row with no address text is `None`.

In [ ]:
import csv
with open(DATA / "addresses_oneline.csv", newline="") as f:
    rows = list(csv.DictReader(f))
parsed = uap.parse_many([r["address"] for r in rows])
for r, p in zip(rows, parsed):
    print(r["id"].rjust(2), p.key() if p else None)

## A dataframe

`doorplate.pandas.parse_column` (and `.polars`) return component columns for a column of one-line addresses. Install the extra: `pip install "doorplate[pandas]"`.

In [ ]:
import pandas as pd
from doorplate.pandas import parse_column

df = pd.read_csv(DATA / "addresses_oneline.csv")
out = df.join(parse_column(df["address"], columns=["street_name", "city", "state", "zip5", "key"]))
out[["id", "addr_street_name", "addr_city", "addr_state", "addr_zip5", "addr_key"]]

## The command line

The `doorplate` CLI does the same on a file: `doorplate parse-file IN.csv --column address -o OUT.csv` appends `addr_`-prefixed columns. Build it with `cargo build --release -p doorplate-cli` (the binary is `target/release/doorplate`).

In [ ]:
import shutil, subprocess
exe = shutil.which("doorplate") or shutil.which("addr")
if exe is None:
    print("doorplate is not on PATH; build it with: cargo build --release -p doorplate-cli")
else:
    r = subprocess.run([exe, "parse-file", str(DATA / "addresses_oneline.csv"), "--column", "address"],
                       capture_output=True, text=True, check=True)
    print(r.stdout.splitlines()[0][:100], "...")
    print(r.stderr.strip())